In [27]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [28]:
from pathlib import Path
import torch

print("PyTorch version:", torch.__version__)
print("Notebook working directory:", Path.cwd())

# If your notebook runs from the notebooks/ folder:
checkpoint_path = Path(
    "../checkpoints/simclr_resnet50_openmmlab_200e.pth"
)

# If it runs from the repository root, use:
# checkpoint_path = Path(
#     "checkpoints/simclr_resnet50_openmmlab_200e.pth"
# )

assert checkpoint_path.is_file(), (
    f"Checkpoint not found: {checkpoint_path.resolve()}"
)

print("Checkpoint:", checkpoint_path.resolve())
print(f"Size: {checkpoint_path.stat().st_size / 1e6:.1f} MB")

PyTorch version: 2.15.0+cu132
Notebook working directory: /sfs/gpfs/tardis/home/xdz3ke/mlia-simclr-brain/notebooks
Checkpoint: /sfs/gpfs/tardis/home/xdz3ke/mlia-simclr-brain/checkpoints/simclr_resnet50_openmmlab_200e.pth
Size: 116.8 MB


In [29]:
%pip install mmengine

Note: you may need to restart the kernel to use updated packages.


In [30]:
import torch

# Only for the checkpoint downloaded directly from OpenMMLab.
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

state_dict = checkpoint["state_dict"]

# Check that we are saving only named tensors.
assert all(
    isinstance(name, str) and torch.is_tensor(value)
    for name, value in state_dict.items()
)

# Preserve the original; write a separate weights-only file.
clean_path = checkpoint_path.with_name(
    checkpoint_path.stem + "_weights_only.pth"
)

torch.save(dict(state_dict), clean_path)

# Confirm the new file loads with the restricted loader.
weights = torch.load(
    clean_path,
    map_location="cpu",
    weights_only=True,
)

print("Saved:", clean_path)
print("Number of weight entries:", len(weights))

for name, value in list(weights.items())[:15]:
    print(name, tuple(value.shape))

Saved: ../checkpoints/simclr_resnet50_openmmlab_200e_weights_only.pth
Number of weight entries: 332
data_preprocessor.mean (3, 1, 1)
data_preprocessor.std (3, 1, 1)
backbone.conv1.weight (64, 3, 7, 7)
backbone.bn1.weight (64,)
backbone.bn1.bias (64,)
backbone.bn1.running_mean (64,)
backbone.bn1.running_var (64,)
backbone.bn1.num_batches_tracked ()
backbone.layer1.0.conv1.weight (64, 64, 1, 1)
backbone.layer1.0.bn1.weight (64,)
backbone.layer1.0.bn1.bias (64,)
backbone.layer1.0.bn1.running_mean (64,)
backbone.layer1.0.bn1.running_var (64,)
backbone.layer1.0.bn1.num_batches_tracked ()
backbone.layer1.0.conv2.weight (64, 64, 3, 3)


In [31]:
%pip install torchvision

Note: you may need to restart the kernel to use updated packages.


In [32]:
from pathlib import Path

import torch
from torch import nn
from torchvision.models import resnet50

checkpoint_path = Path(
    "../checkpoints/simclr_resnet50_openmmlab_200e_weights_only.pth"
)

weights = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)

# Keep only the encoder weights.
# Remove OpenMMLab's "backbone." prefix to match torchvision's names.
backbone_weights = {
    key.removeprefix("backbone."): value
    for key, value in weights.items()
    if key.startswith("backbone.")
}

encoder = resnet50(weights=None)

# Return image features instead of predictions for ImageNet classes.
encoder.fc = nn.Identity()

# Require every expected encoder tensor to match.
result = encoder.load_state_dict(backbone_weights, strict=True)
print(result)

# Freeze the encoder for our initial feature-extraction experiment.
encoder.requires_grad_(False)
encoder.eval()

<All keys matched successfully>


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [33]:
# Two artificial RGB images, each 224 × 224.
dummy_images = torch.rand(2, 3, 224, 224)

with torch.inference_mode():
    features = encoder(dummy_images)

print("Input shape:", tuple(dummy_images.shape))
print("Feature shape:", tuple(features.shape))
print("All values finite:", torch.isfinite(features).all().item())

assert features.shape == (2, 2048)
assert torch.isfinite(features).all()

Input shape: (2, 3, 224, 224)
Feature shape: (2, 2048)
All values finite: True


In [34]:
for key in ("data_preprocessor.mean", "data_preprocessor.std"):
    print(key, weights[key].flatten().tolist())

data_preprocessor.mean [123.67500305175781, 116.27999877929688, 103.52999877929688]
data_preprocessor.std [58.39500045776367, 57.119998931884766, 57.375]


In [35]:
import numpy as np

train_images = np.load("../data/brain_train_image_final.npy")
print(train_images.shape)

(1657, 2, 100, 76)


In [36]:
import torch
import torch.nn.functional as F

# Select plane 0 from four training examples.
# Original shape: (N, 2, 100, 76)
images = torch.as_tensor(
    train_images[:4, 0],
    dtype=torch.float32,
).unsqueeze(1)

# Shape: (4, 1, 100, 76)
assert torch.isfinite(images).all()
assert images.min() >= 0 and images.max() <= 1

# Resize the grayscale images.
images = F.interpolate(
    images,
    size=(224, 224),
    mode="bilinear",
    align_corners=False,
    antialias=True,
)

# Repeat the grayscale channel to make three channels.
images = images.repeat(1, 3, 1, 1)

# Convert checkpoint normalization values to the 0–1 scale.
mean = weights["data_preprocessor.mean"].reshape(1, 3, 1, 1) / 255.0
std = weights["data_preprocessor.std"].reshape(1, 3, 1, 1) / 255.0

images = (images - mean) / std

# Extract features without changing the encoder.
device = next(encoder.parameters()).device
encoder.eval()

with torch.inference_mode():
    features = encoder(images.to(device)).cpu()

print("Preprocessed image shape:", tuple(images.shape))
print("Feature shape:", tuple(features.shape))
print("All features finite:", torch.isfinite(features).all().item())

assert features.shape == (4, 2048)
assert torch.isfinite(features).all()

Preprocessed image shape: (4, 3, 224, 224)
Feature shape: (4, 2048)
All features finite: True


In [37]:
# Start of First Linear Probe
def preprocess_mri(batch, mean, std):
    """Convert a batch of grayscale images (B, H, W) into encoder inputs."""
    batch = torch.as_tensor(batch, dtype=torch.float32).unsqueeze(1)

    batch = F.interpolate(
        batch,
        size=(224, 224),
        mode="bilinear",
        align_corners=False,
        antialias=True,
    )

    batch = batch.repeat(1, 3, 1, 1)
    return (batch - mean) / std

In [38]:
mean = weights["data_preprocessor.mean"].reshape(1, 3, 1, 1) / 255.0
std = weights["data_preprocessor.std"].reshape(1, 3, 1, 1) / 255.0

# Use the device where your encoder is already located.
device = next(encoder.parameters()).device
encoder.requires_grad_(False)
encoder.eval()

batch_size = 32
feature_batches = []

with torch.inference_mode():
    for start in range(0, len(train_images), batch_size):
        # Select plane 0 for this batch.
        batch = train_images[start:start + batch_size, 1]

        inputs = preprocess_mri(batch, mean, std).to(device)
        batch_features = encoder(inputs).cpu()

        feature_batches.append(batch_features)

        if start % (batch_size * 10) == 0:
            print(f"Processed {min(start + batch_size, len(train_images))}"
                  f" / {len(train_images)} images")

all_features = torch.cat(feature_batches, dim=0).numpy()

print("Feature matrix:", all_features.shape)
print("All features finite:", np.isfinite(all_features).all())

assert all_features.shape == (len(train_images), 2048)
assert np.isfinite(all_features).all()

Processed 32 / 1657 images
Processed 352 / 1657 images
Processed 672 / 1657 images
Processed 992 / 1657 images
Processed 1312 / 1657 images
Processed 1632 / 1657 images
Feature matrix: (1657, 2048)
All features finite: True


In [39]:
# Load split 
with np.load("../splits/train_val_seed42.npz") as split:
    train_idx = split["train_idx"]
    val_idx = split["val_idx"]
    y = split["labels"]

assert len(all_features) == len(y)
assert all_features.shape == (len(y), 2048)
assert np.isfinite(all_features).all()

X_train = all_features[train_idx]
X_val = all_features[val_idx]
y_train = y[train_idx]
y_val = y[val_idx]

print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)

Training features: (1325, 2048)
Validation features: (332, 2048)


In [40]:
# Train Linear Classifier 
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

probe = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        solver="lbfgs",
        C=1.0,
        max_iter=2000,
    ),
)

probe.fit(X_train, y_train)

print("Linear classifier trained.")

Linear classifier trained.


In [41]:
# Validation Performance Measuring

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

train_pred = probe.predict(X_train)
val_pred = probe.predict(X_val)
val_probs = probe.predict_proba(X_val)

classes = probe.named_steps["logisticregression"].classes_

print(f"Training accuracy:   {accuracy_score(y_train, train_pred):.4f}")
print(f"Validation accuracy: {accuracy_score(y_val, val_pred):.4f}")

print("\nValidation classification report:")
print(classification_report(
    y_val,
    val_pred,
    labels=classes,
    digits=4,
    zero_division=0,
))

val_auc = roc_auc_score(
    y_val,
    val_probs,
    labels=classes,
    multi_class="ovr",
    average="macro",
)
print(f"Validation macro AUROC (one-vs-rest): {val_auc:.4f}")

print("\nConfusion matrix — rows: actual, columns: predicted")
print("Class order:", classes)
print(confusion_matrix(y_val, val_pred, labels=classes))

assert np.intersect1d(train_idx, val_idx).size == 0

assert np.array_equal(X_train, all_features[train_idx])
assert np.array_equal(X_val, all_features[val_idx])
assert np.array_equal(y_train, y[train_idx])
assert np.array_equal(y_val, y[val_idx])

print("Split and feature alignment checks passed.")

Training accuracy:   1.0000
Validation accuracy: 0.6837

Validation classification report:
              precision    recall  f1-score   support

           0     0.7176    0.7642    0.7402       123
           1     0.7100    0.7474    0.7282        95
           2     0.6139    0.5439    0.5767       114

    accuracy                         0.6837       332
   macro avg     0.6805    0.6852    0.6817       332
weighted avg     0.6798    0.6837    0.6806       332

Validation macro AUROC (one-vs-rest): 0.8252

Confusion matrix — rows: actual, columns: predicted
Class order: [0 1 2]
[[94  7 22]
 [ 7 71 17]
 [30 22 62]]
Split and feature alignment checks passed.


In [42]:
import numpy as np
import hashlib

raw_images = np.load("../data/brain_train_image_final.npy")

def image_hash(image):
    return hashlib.sha256(
        np.ascontiguousarray(image).tobytes()
    ).hexdigest()

plane1_hashes = [
    image_hash(image)
    for image in raw_images[:, 1]
]

print("Unique plane-1 images:", len(set(plane1_hashes)))

for class_id in np.unique(y):
    class_indices = np.flatnonzero(y == class_id)
    unique_count = len({plane1_hashes[i] for i in class_indices})
    print(f"Class {class_id}: {unique_count} unique plane-1 images")

training_hashes = {plane1_hashes[i] for i in train_idx}

overlap = sum(
    plane1_hashes[i] in training_hashes
    for i in val_idx
)

print("Plane-1 validation examples duplicated in training:", overlap)

Unique plane-1 images: 1657
Class 0: 612 unique plane-1 images
Class 1: 477 unique plane-1 images
Class 2: 568 unique plane-1 images
Plane-1 validation examples duplicated in training: 0
